# BAB 3: DATA TRANSFORMATION

Data Transformation adalah proses mengubah format, struktur, atau rentang nilai data mentah yang telah bersih menjadi bentuk baru yang siap dikonsumsi oleh algoritma Machine Learning / AI. Langkah ini wajib dilakukan karena algoritma AI murni bekerja menggunakan perhitungan matematika.

---

## A. Feature Encoding (Konversi Teks ke Angka)
Algoritma machine learning tidak bisa membaca teks string secara langsung. Oleh karena itu, fitur bertipe kategori/teks harus diubah menjadi representasi angka numerik.

1. **Ordinal Encoding**
   * **Konsep:** Digunakan untuk kolom kategori yang memiliki tingkatan, urutan, atau hierarki nilai yang jelas (ranked data).
   * **Penerapan Pandas:** Menggunakan fungsi `.map()` dengan bantuan *dictionary* kustom.
   * **Contoh:** Mengubah status merokok (`Tidak Pernah` ➜ 0, `Jarang` ➜ 1, `Sering` ➜ 2).

2. **One-Hot Encoding**
   * **Konsep:** Digunakan untuk kolom kategori nominal, yaitu nilai kategori yang setara dan tidak memiliki tingkatan/hierarki khusus.
   * **Penerapan Pandas:** Menggunakan fungsi `pd.get_dummies(df, columns=['Nama_Kolom'], dtype=int)`.
   * **Contoh:** Mengubah kolom `Jenis_Kelamin` (`Pria`, `Wanita`) menjadi dua kolom biner baru bernilai 0 atau 1.

> ⚠️ **Poin Penting (Curse of Dimensionality):**
> Hindari menggunakan One-Hot Encoding pada fitur yang memiliki **Kardinalitas Tinggi** (kolom teks yang punya ratusan/ribuan nilai unik, seperti Alamat, Kota, atau ID). Melakukan hal ini akan membuat DataFrame membengkak ke samping secara ekstrem, menghabiskan memori RAM, dan memperlambat proses training AI.

---

## B. Feature Scaling (Penyetaraan Rentang Numerik)
Ketika fitur numerik memiliki rentang nilai yang jomplang (misal: usia berkisar belasan, sedangkan pendapatan berkisar jutaan), model AI akan cenderung "terkecoh" dan menganggap fitur dengan angka besar jauh lebih penting. Scaling menyetarakan bobot rentang tersebut.

> ❗ **Aturan Urutan Operasi:** 
> Pastikan proses **Encoding dan Drop kolom teks diselesaikan terlebih dahulu**. Proses Scaling murni dilakukan pada DataFrame yang hanya menyisakan kolom-kolom target numerik agar komputasi Pandas tidak mengalami *error*.

1. **Min-Max Scaling (Normalisasi)**
   * **Definisi:** Mengubah rentang data secara mutlak menjadi berada di antara angka **0 hingga 1**.
   * **Formula Pandas:** `(df - df.min()) / (df.max() - df.min())`
   * **Karakteristik:** Sangat sensitif terhadap data ekstrem (*outlier*). Jika ada satu nilai yang melompat terlalu tinggi, nilai normal lainnya akan "tertekan" (*squeezed*) berkumpul sangat dekat dengan angka 0.
   * **Kapan Digunakan:** Cocok untuk data yang tidak berdistribusi normal, batas minimum-maksimumnya diketahui pasti (seperti nilai piksel gambar 0–255), atau saat menggunakan algoritma berbasis jarak seperti KNN dan Neural Networks.

2. **Standardization (Standardisasi)**
   * **Definisi:** Menggeser seluruh data agar titik tengahnya (**Rata-rata / Mean = 0**) dan sebaran datanya (**Standar Deviasi = 1**).
   * **Formula Pandas:** `(df - df.mean()) / df.std()`
   * **Karakteristik:** Angka hasil scaling bisa bernilai minus (artinya di bawah rata-rata) atau lebih dari 1. Jauh lebih kebal (*robust*) terhadap efek data ekstrem (*outlier*).
   * **Kapan Digunakan:** Sangat direkomendasikan jika data berdistribusi normal (kurva lonceng), memiliki banyak pencilan/outlier jomplang, atau saat menggunakan algoritma berbasis linear (Linear/Logistic Regression, SVM, PCA).

---

## C. Structural Transformation (Pembersihan Struktur Data)
Sebelum melangkah ke perhitungan matematika, struktur tabel harus dirapikan untuk memastikan konsistensi data dan menghindari *runtime error*.

1. **Eliminasi Kolom (`.drop()`)**
   Menyingkirkan kolom teks asli yang sudah di-encode, atau menghapus kolom identitas unik (seperti nama pasien atau nomor ID) yang tidak memiliki pola prediktif bagi model AI.

2. **Standardisasi Nama Kolom (`.rename()`)**
   Mengubah nama-nama kolom menjadi format standar *snake_case* (huruf kecil semua dan spasi diganti dengan garis bawah `_`). Hal ini penting agar nama kolom kompatibel saat diproses oleh library machine learning tingkat lanjut.

3. **Transformasi Baris Custom / Pembuatan Kategori Baru (`.apply()`)**
   Membuat fungsi kustom (Python *function*) yang berisi logika kondisi `if-elif-else`, kemudian menerapkannya ke DataFrame menggunakan `.apply(nama_fungsi, axis=1)`. Angka `axis=1` wajib ditulis agar Pandas membaca data per baris, bukan per kolom.

---


In [24]:
import pandas as pd
import numpy as np

In [25]:
# Dataset
data = {
    "Nama": ["Ali", "Budi", "Cici", "Dedi"],
    "Status_Merokok": ["Sering", "Tidak Pernah", "Jarang", "Tidak Pernah"],
    "Jenis_Kelamin": ["Pria", "Pria", "Wanita" , "Pria"],
    "Usia": [20, 40, 50, 22],
    "Pendapatan": [3000000, 8000000, 40000000, 15000000]
}

df = pd.DataFrame(data)

In [26]:
# Feature Encoding (Mengubah Teks menjadi Angka)

# Ordinal Encoding
# Keunikan datanya bisa bertingkat (1, 2, atau 3)
mapping_rokok = {"Tidak Pernah" : 0, "Jarang" : 1, "Sering" : 2}
df["Status_Merokok_Encoded"] = df["Status_Merokok"].map(mapping_rokok)
display(df)

# One-hot Encoding
# Keunikan datanya bersifat 0 atau 1
# .get_dummies(data, columns, data type)
df_encoded = pd.get_dummies(df, columns=["Jenis_Kelamin"], dtype=int)
display(df_encoded)

,Nama,Status_Merokok,Jenis_Kelamin,Usia,Pendapatan,Status_Merokok_Encoded
0,Ali,Sering,Pria,20,3000000,2
1,Budi,Tidak Pernah,Pria,40,8000000,0
2,Cici,Jarang,Wanita,50,40000000,1
3,Dedi,Tidak Pernah,Pria,22,15000000,0


,Nama,Status_Merokok,Usia,Pendapatan,Status_Merokok_Encoded,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,Ali,Sering,20,3000000,2,1,0
1,Budi,Tidak Pernah,40,8000000,0,1,0
2,Cici,Jarang,50,40000000,1,0,1
3,Dedi,Tidak Pernah,22,15000000,0,1,0


In [27]:
# Feature Scaling

# Buang kolom yang tidak integer
df_numerik = df_encoded.drop(columns=["Nama", "Status_Merokok"])
# Menentukan kolom target scaling
target_scaling = ["Usia", "Pendapatan"]
# Copy DataFrame untuk eksperimen
df_minmax = df_numerik.copy()
df_standard = df_numerik.copy()

# 1. Min-Max Scaling
# mengubah semua angka dalam kolom menjadi rentan 0 sampai 1
# Sangat disarankan ketika ingin mempertahankan batas atas dan batas bawah
df_minmax[target_scaling] = (df_numerik[target_scaling] - df_numerik[target_scaling].min()) / (df_numerik[target_scaling].max() - df_numerik[target_scaling].min())
display(df_minmax)

# 2. Standardization
# mengubah semua angka dalam kolom supaya berpusat di 0 
# Sangat disarankan ketika terdapat yang outliner (pencilan)
df_standard[target_scaling] = (df_numerik[target_scaling] - df_numerik[target_scaling].mean()) / df_numerik[target_scaling].std()
display(df_standard)

,Usia,Pendapatan,Status_Merokok_Encoded,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,0.000000,0.000000,2,1,0
1,0.666667,0.135135,0,1,0
2,1.000000,1.000000,1,0,1
3,0.066667,0.324324,0,1,0


,Usia,Pendapatan,Status_Merokok_Encoded,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,-0.898513,-0.822091,2,1,0
1,0.483814,-0.517613,0,1,0
2,1.174978,1.431048,1,0,1
3,-0.760280,-0.091343,0,1,0


In [31]:
# Structural Transformation

# 1. Menghapus Kolom/Baris
# digunakan untuk membuang fitur yang tidak berguna dalam pemisahan antara fitur dan target
# .drop()
X = df_encoded.drop(columns=["Nama", "Status_Merokok"])
display(X)

# 2. Mengubah Nama Kolom
# digunakan untuk mengubah nama kolom agar bersih 
df_clean_names = df_encoded.rename(columns={
    "Nama" : "Nama Lengkap",
    "Pendapatan" : "Gaji"
})
display(df_clean_names)

# 3. Transformasi Baris Custom Kategori
# digunakan untuk memungkinkan menerapkan fungsi Python ke dalam tiap kolom atau barus
df_encoded["Kategori_Usia"] = df_encoded["Usia"].apply(lambda x: "Muda" if x < 30 else "Tua")
display(df_encoded[["Nama", "Usia", "Kategori_Usia"]])

,Usia,Pendapatan,Status_Merokok_Encoded,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita,Kategori_Usia
0,20,3000000,2,1,0,Muda
1,40,8000000,0,1,0,Tua
2,50,40000000,1,0,1,Tua
3,22,15000000,0,1,0,Muda


,Nama Lengkap,Status_Merokok,Usia,Gaji,Status_Merokok_Encoded,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita,Kategori_Usia
0,Ali,Sering,20,3000000,2,1,0,Muda
1,Budi,Tidak Pernah,40,8000000,0,1,0,Tua
2,Cici,Jarang,50,40000000,1,0,1,Tua
3,Dedi,Tidak Pernah,22,15000000,0,1,0,Muda


,Nama,Usia,Kategori_Usia
0,Ali,20,Muda
1,Budi,40,Tua
2,Cici,50,Tua
3,Dedi,22,Muda
